In [ ]:
# =========================================================
# KAGGLE READY CHECK
# =========================================================

from pathlib import Path
import os
import torch

IS_KAGGLE = Path("/kaggle/input").exists()

print(f"[INFO] Kaggle environment: {IS_KAGGLE}")
print(f"[INFO] CUDA available: {torch.cuda.is_available()}")
print(f"[INFO] GPU count: {torch.cuda.device_count()}")

unsupported_gpus = []

for gpu_idx in range(torch.cuda.device_count()):
    gpu_name = torch.cuda.get_device_name(gpu_idx)
    capability = torch.cuda.get_device_capability(gpu_idx)

    print(f"[INFO] GPU {gpu_idx}: {gpu_name} | CUDA capability: sm_{capability[0]}{capability[1]}")

    if capability[0] < 7:
        unsupported_gpus.append(gpu_name)

if unsupported_gpus:
    print("\n[ERROR] This runtime has an old GPU for the installed PyTorch build.")
    print(f"[ERROR] Unsupported GPU(s): {unsupported_gpus}")
    print("[FIX] In Kaggle Settings, choose Accelerator: GPU T4 x2, then restart the session.")

if IS_KAGGLE:
    print("\n[INFO] Kaggle input datasets:")
    for item in Path("/kaggle/input").iterdir():
        print(f"- {item}")
else:
    print("\n[INFO] Running locally. Kaggle paths will be ignored.")

In [ ]:
# =========================================================
# NOTE POUR EXECUTION LOCALE RTX 3050
# =========================================================

# Ce notebook est maintenant adapt? pour ton PC local avec RTX 3050.
# Objectif: ?viter de t?l?charger/copier plus que n?cessaire.

# Strat?gie dataset:
# - Cherche d'abord des datasets d?j? pr?sents sur ton disque.
# - Si AUTO_DOWNLOAD_KAGGLE=True, utilise l'API Kaggle en mode ?chantillonn?.
# - Le mode ?chantillonn? t?l?charge au maximum MAX_IMAGES_PER_CLASS images par classe.
# - Il ne t?l?charge pas le dataset Kaggle complet sauf si ALLOW_FULL_KAGGLE_DOWNLOAD=True.

# Pr?-requis pour t?l?chargement Kaggle local:
# 1. Installer/configurer Kaggle API une seule fois.
# 2. Placer kaggle.json dans ~/.kaggle/kaggle.json
# 3. Ou mettre les datasets d?j? extraits dans ./datasets/kaggle_cache/

# Pour ?conomiser ton internet:
# - Garde AUTO_DOWNLOAD_KAGGLE=True seulement si tu veux t?l?charger les images manquantes.
# - Une fois les images t?l?charg?es, remets AUTO_DOWNLOAD_KAGGLE=False.
# - Le cache local sera r?utilis? aux prochains lancements.

In [ ]:
# =========================================================
# IMPORTS + CONFIGURATION
# =========================================================

import os
import sys
import json
import itertools
import hashlib
import shutil
import subprocess
import zipfile
from pathlib import Path

import numpy as np
import torch
import torchvision
import torch.nn as nn
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    from sklearn.metrics import (
        auc,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_curve
    )
    from sklearn.preprocessing import label_binarize
except ImportError:
    print("[INFO] Installing scikit-learn...")
    os.system(f"{sys.executable} -m pip install -q scikit-learn")
    from sklearn.metrics import (
        auc,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_curve
    )
    from sklearn.preprocessing import label_binarize

try:
    from torchinfo import summary
except ImportError:
    summary = None
    print("[WARN] torchinfo is not installed. Model summary will be skipped.")

SEED = 42
EPOCHS = 20
TRAIN_RATIO = 0.8
LEARNING_RATE = 1e-4
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_MIN_DELTA = 1e-4
FINE_TUNE_LAST_BLOCKS = 2
MAX_IMAGES_PER_CLASS = 400
SELECTED_CLASSES = None
DATASET_BUILD_VERSION = "direct_class_sampling_v3"
RUN_MODEL_SUMMARY = False
COMPUTE_ROC_AUC = True
ROC_MAX_SAMPLES = 15000
MAX_CONFUSION_ANNOTATION_CLASSES = 35

# Optional final override. If you already have one merged ImageFolder dataset,
# set DATASET_ROOT and the Kaggle/Drive merge logic will be skipped.
DATASET_ROOT = None

# Kaggle datasets to import locally. Download mode is local-first and sampled.
KAGGLE_DATASET_SLUGS = [
    "kmader/food41",
    "yassineelmir/tunisian-cuisine",
]

KAGGLE_CACHE_DIR = Path("datasets/kaggle_cache")
KAGGLE_SAMPLED_ROOT = KAGGLE_CACHE_DIR / "sampled"
KAGGLE_FULL_ROOT = KAGGLE_CACHE_DIR / "full"
KAGGLE_TMP_DIR = KAGGLE_CACHE_DIR / "tmp"

# Local folders checked before any download. Add your own extracted dataset paths here.
KAGGLE_DATASET_ROOTS = [
    KAGGLE_SAMPLED_ROOT / "kmader__food41",
    KAGGLE_SAMPLED_ROOT / "yassineelmir__tunisian-cuisine",
    KAGGLE_FULL_ROOT / "kmader__food41",
    KAGGLE_FULL_ROOT / "yassineelmir__tunisian-cuisine",
    Path("food41"),
    Path("tunisian-cuisine"),
    Path("datasets/food41"),
    Path("datasets/tunisian-cuisine"),
    Path("food101_60"),
    Path("new_dataset"),
]
KAGGLE_DATASET_ROOT = None

# Internet-saving Kaggle download controls.
AUTO_DOWNLOAD_KAGGLE = True
KAGGLE_DOWNLOAD_MODE = "sample"
ALLOW_FULL_KAGGLE_DOWNLOAD = False
KAGGLE_DOWNLOAD_QUIET = False

# Google Drive is optional and disabled by default.
DRIVE_FOLDER_URL = ""  # Configure your own dataset locally.
DRIVE_ZIP_URL = ""
DRIVE_DATASET_ROOT = None
ALLOW_SMALL_DRIVE_FOLDER_DOWNLOAD = False

# Optional zip file override for Kaggle/local zip datasets.
ZIP_DATASET_PATH = None
AUTO_EXTRACT_ZIP = True
MAX_AUTO_ZIPS = 2

COMBINE_DATASETS = True
REBUILD_COMBINED_DATASET = False
LINK_DATASETS_INSTEAD_OF_COPY = True

IS_KAGGLE = Path("/kaggle/input").exists()
INPUT_DIR = Path("/kaggle/input") if IS_KAGGLE else Path(".")
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")
MODEL_DIR = WORKING_DIR / "models"
PLOT_DIR = WORKING_DIR
COMBINED_DATASET_ROOT = WORKING_DIR / "combined_dataset"
DRIVE_DOWNLOAD_DIR = WORKING_DIR / "drive_dataset"

MODEL_PATH = MODEL_DIR / "effnetb2_food101.pth"
HISTORY_PATH = MODEL_DIR / "history.json"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

NUM_GPUS = torch.cuda.device_count()
device = "cuda" if torch.cuda.is_available() else "cpu"

MIN_CUDA_CAPABILITY_MAJOR = 7
unsupported_gpus = []

if device == "cuda":
    for gpu_idx in range(NUM_GPUS):
        gpu_name = torch.cuda.get_device_name(gpu_idx)
        capability = torch.cuda.get_device_capability(gpu_idx)

        print(f"[INFO] GPU {gpu_idx}: {gpu_name} | CUDA capability: sm_{capability[0]}{capability[1]}")

        if capability[0] < MIN_CUDA_CAPABILITY_MAJOR:
            unsupported_gpus.append(f"{gpu_name} (sm_{capability[0]}{capability[1]})")

if unsupported_gpus:
    raise RuntimeError(
        "This runtime GPU is not compatible with the installed PyTorch build: "
        f"{unsupported_gpus}. In Kaggle Settings, select Accelerator: GPU T4 x2, "
        "then restart the session and run the notebook again."
    )

# RTX 3050 friendly defaults. Increase BATCH_SIZE only after a stable run.
GPU_MEMORY_GB = 0
if device == "cuda":
    GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

BATCH_SIZE = 16 if GPU_MEMORY_GB >= 5.5 else 8
NUM_WORKERS = 0 if os.name == "nt" else min(2, os.cpu_count() or 2)
PERSISTENT_WORKERS = False
PIN_MEMORY = device == "cuda"
USE_AMP = device == "cuda"
AMP_DEVICE_TYPE = "cuda" if device == "cuda" else "cpu"

if device == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"[INFO] Using device: {device}")
print(f"[INFO] GPU count: {NUM_GPUS}")
print(f"[INFO] GPU memory: {GPU_MEMORY_GB:.1f} GB")
print(f"[INFO] Batch size: {BATCH_SIZE}")
print(f"[INFO] Num workers: {NUM_WORKERS}")
print(f"[INFO] Persistent workers: {PERSISTENT_WORKERS}")
print(f"[INFO] Mixed precision AMP: {USE_AMP}")
print(f"[INFO] AMP device type: {AMP_DEVICE_TYPE}")
print(f"[INFO] Working directory: {WORKING_DIR}")

In [ ]:
# =========================================================
# DATASET HELPERS
# =========================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def set_seeds(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)


def is_imagefolder_root(path, min_classes=1):
    path = Path(path)

    if not path.is_dir():
        return False

    class_dirs = [
        child
        for child in path.iterdir()
        if child.is_dir() and class_folder_has_images(child)
    ]

    return len(class_dirs) >= min_classes


def find_imagefolder_inside(root, min_classes=1):
    root = Path(root)

    if is_imagefolder_root(root, min_classes=min_classes):
        return root

    if not root.exists():
        return None

    for candidate in root.rglob("*"):
        if candidate.is_dir() and is_imagefolder_root(candidate, min_classes=min_classes):
            return candidate

    return None


def unique_paths(paths):
    seen = set()
    unique = []

    for path in paths:
        path = Path(path)
        key = str(path.resolve()) if path.exists() else str(path)

        if key not in seen:
            seen.add(key)
            unique.append(path)

    return unique


def kaggle_slug_dirname(slug):
    return slug.replace("/", "__")


def kaggle_sampled_dir(slug):
    return KAGGLE_SAMPLED_ROOT / kaggle_slug_dirname(slug)


def kaggle_full_dir(slug):
    return KAGGLE_FULL_ROOT / kaggle_slug_dirname(slug)


def import_kaggle_api():
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ImportError:
        print("[INFO] Installing kaggle package...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
        from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    return api


def kaggle_file_ref(file_obj):
    return getattr(file_obj, "name", None) or getattr(file_obj, "ref", None)


def class_name_from_file_ref(file_ref):
    path = Path(file_ref)

    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        return None

    if len(path.parts) < 2:
        return None

    class_name = path.parts[-2]

    if class_name.lower() in {"images", "image", "img", "data", "train", "test", "valid", "validation"}:
        return None

    return class_name


def selected_kaggle_refs(api, slug):
    print(f"[INFO] Listing Kaggle dataset files: {slug}")
    dataset_files = api.dataset_list_files(slug).files
    allowed_classes = selected_classes_set()
    selected_by_class = {}
    totals_by_class = {}
    rng_by_class = {}

    for file_obj in dataset_files:
        file_ref = kaggle_file_ref(file_obj)

        if not file_ref:
            continue

        class_name = class_name_from_file_ref(file_ref)

        if class_name is None:
            continue

        if allowed_classes is not None and class_name not in allowed_classes:
            continue

        totals_by_class[class_name] = totals_by_class.get(class_name, 0) + 1
        seen_count = totals_by_class[class_name]
        selected = selected_by_class.setdefault(class_name, [])

        if class_name not in rng_by_class:
            stable_hash = hashlib.md5(f"{SEED}_{slug}_{class_name}".encode("utf-8")).hexdigest()
            rng_by_class[class_name] = np.random.default_rng(int(stable_hash[:8], 16))

        if MAX_IMAGES_PER_CLASS is None or len(selected) < MAX_IMAGES_PER_CLASS:
            selected.append((seen_count, file_ref))
        else:
            replace_idx = rng_by_class[class_name].integers(0, seen_count)

            if replace_idx < MAX_IMAGES_PER_CLASS:
                selected[replace_idx] = (seen_count, file_ref)

    for class_name in sorted(totals_by_class.keys()):
        print(
            f"[INFO] Kaggle {slug} / class '{class_name}': "
            f"selected {len(selected_by_class.get(class_name, []))}/{totals_by_class[class_name]} files"
        )

    return selected_by_class


def download_single_kaggle_file(api, slug, file_ref, target_path):
    target_path = Path(target_path)

    if target_path.exists():
        return

    tmp_id = hashlib.md5(f"{slug}_{file_ref}".encode("utf-8")).hexdigest()
    tmp_dir = KAGGLE_TMP_DIR / kaggle_slug_dirname(slug) / tmp_id
    safe_reset_directory(tmp_dir)

    try:
        api.dataset_download_file(
            slug,
            file_name=file_ref,
            path=str(tmp_dir),
            force=False,
            quiet=KAGGLE_DOWNLOAD_QUIET
        )

        downloaded_images = [
            candidate
            for candidate in tmp_dir.rglob("*")
            if candidate.is_file() and candidate.suffix.lower() in IMAGE_EXTENSIONS
        ]

        if not downloaded_images:
            print(f"[WARN] No image downloaded for Kaggle file: {file_ref}")
            return

        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(downloaded_images[0], target_path)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


def download_kaggle_dataset_sample(api, slug):
    output_root = kaggle_sampled_dir(slug)

    if is_imagefolder_root(output_root, min_classes=1):
        print(f"[INFO] Reusing sampled Kaggle cache: {output_root}")
        return output_root

    selected_by_class = selected_kaggle_refs(api, slug)

    if not selected_by_class:
        message = (
            f"Could not sample individual image files from Kaggle dataset {slug}. "
            "The dataset may be packaged as one archive."
        )

        if not ALLOW_FULL_KAGGLE_DOWNLOAD:
            print(f"[WARN] {message}")
            print("[WARN] Full Kaggle download is disabled to protect your slow internet.")
            return None

        return download_full_kaggle_dataset(api, slug)

    output_root.mkdir(parents=True, exist_ok=True)

    for class_name in sorted(selected_by_class.keys()):
        for sample_idx, (seen_count, file_ref) in enumerate(sorted(selected_by_class[class_name])):
            suffix = Path(file_ref).suffix.lower() or ".jpg"
            ref_hash = hashlib.md5(file_ref.encode("utf-8")).hexdigest()[:12]
            target_path = output_root / class_name / f"{sample_idx:08d}_{seen_count:08d}_{ref_hash}{suffix}"
            download_single_kaggle_file(api, slug, file_ref, target_path)

    if not is_imagefolder_root(output_root, min_classes=1):
        print(f"[WARN] Sampled Kaggle download did not create an ImageFolder dataset: {output_root}")
        return None

    return output_root


def download_full_kaggle_dataset(api, slug):
    output_root = kaggle_full_dir(slug)
    output_root.mkdir(parents=True, exist_ok=True)

    if find_imagefolder_inside(output_root, min_classes=1) is not None:
        print(f"[INFO] Reusing full Kaggle cache: {output_root}")
        return output_root

    print(f"[INFO] Downloading full Kaggle dataset: {slug}")
    api.dataset_download_files(
        slug,
        path=str(output_root),
        unzip=True,
        quiet=KAGGLE_DOWNLOAD_QUIET
    )

    return output_root


def prepare_local_kaggle_cache():
    if IS_KAGGLE or not AUTO_DOWNLOAD_KAGGLE:
        return

    existing_roots = [
        root for root in candidate_kaggle_dataset_roots()
        if find_imagefolder_inside(root, min_classes=1) is not None
    ]

    if existing_roots:
        print("[INFO] Local Kaggle datasets already found; no download needed.")
        return

    try:
        api = import_kaggle_api()
    except Exception as error:
        print(f"[WARN] Kaggle API is not ready: {error}")
        print("[WARN] Put kaggle.json in ~/.kaggle/ or disable AUTO_DOWNLOAD_KAGGLE.")
        return

    for slug in KAGGLE_DATASET_SLUGS:
        if KAGGLE_DOWNLOAD_MODE == "sample":
            download_kaggle_dataset_sample(api, slug)
        elif KAGGLE_DOWNLOAD_MODE == "full":
            if ALLOW_FULL_KAGGLE_DOWNLOAD:
                download_full_kaggle_dataset(api, slug)
            else:
                print(f"[WARN] Full download skipped for {slug}; ALLOW_FULL_KAGGLE_DOWNLOAD=False")
        else:
            print(f"[WARN] Unknown KAGGLE_DOWNLOAD_MODE={KAGGLE_DOWNLOAD_MODE}")


def candidate_kaggle_dataset_roots():
    candidates = []

    for slug in KAGGLE_DATASET_SLUGS:
        candidates.extend([
            kaggle_sampled_dir(slug),
            kaggle_full_dir(slug)
        ])

    for configured_root in KAGGLE_DATASET_ROOTS:
        candidates.append(Path(configured_root))

    if KAGGLE_DATASET_ROOT is not None:
        candidates.append(Path(KAGGLE_DATASET_ROOT))

    if IS_KAGGLE:
        for dataset_dir in INPUT_DIR.iterdir():
            if not dataset_dir.is_dir():
                continue

            candidates.extend([
                dataset_dir,
                dataset_dir / "food101_60",
                dataset_dir / "food-101" / "images",
                dataset_dir / "images",
                dataset_dir / "train",
                dataset_dir / "data"
            ])

            for child in dataset_dir.iterdir():
                if child.is_dir():
                    candidates.append(child)
    else:
        candidates.extend([
            Path("food41"),
            Path("tunisian-cuisine"),
            Path("datasets/food41"),
            Path("datasets/tunisian-cuisine"),
            Path("food101_60"),
            Path("new_dataset")
        ])

    return unique_paths(candidates)


def candidate_zip_files():
    if ZIP_DATASET_PATH is not None:
        return [Path(ZIP_DATASET_PATH)]

    search_roots = [Path(".")]

    if IS_KAGGLE:
        search_roots.append(INPUT_DIR)

    zip_files = []

    for root in search_roots:
        if root.exists():
            zip_files.extend(sorted(root.rglob("*.zip")))

    return zip_files[:MAX_AUTO_ZIPS]


def find_dataset_root_in_extracted_zips():
    if not AUTO_EXTRACT_ZIP:
        return None

    for zip_path in candidate_zip_files():
        if not zip_path.exists():
            continue

        extract_dir = WORKING_DIR / zip_path.stem
        extract_dir.mkdir(parents=True, exist_ok=True)

        if not any(extract_dir.iterdir()):
            print(f"[INFO] Extracting zip dataset: {zip_path}")

            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(extract_dir)

        extracted_root = find_imagefolder_inside(extract_dir, min_classes=1)

        if extracted_root is not None:
            return extracted_root

    return None


def find_kaggle_dataset_roots():
    detected_roots = []
    seen = set()

    for candidate in candidate_kaggle_dataset_roots():
        detected_root = find_imagefolder_inside(candidate, min_classes=1)

        if detected_root is None:
            continue

        key = str(detected_root.resolve())

        if key not in seen:
            seen.add(key)
            detected_roots.append(detected_root)

    if not detected_roots:
        zipped_root = find_dataset_root_in_extracted_zips()

        if zipped_root is not None:
            detected_roots.append(zipped_root)

    return detected_roots


def import_gdown():
    try:
        import gdown
        return gdown
    except ImportError:
        print("[INFO] Installing gdown...")
        os.system(f"{sys.executable} -m pip install -q gdown")
        import gdown
        return gdown


def download_drive_zip(gdown, url):
    DRIVE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    drive_file = WORKING_DIR / "drive_dataset.zip"

    print(f"[INFO] Downloading Google Drive ZIP to: {drive_file}")
    downloaded = gdown.download(
        url=url,
        output=str(drive_file),
        quiet=False,
        fuzzy=True
    )

    if downloaded is None:
        raise RuntimeError(
            "Google Drive ZIP download failed. Make sure the link is public and Kaggle Internet is enabled."
        )

    downloaded_path = Path(downloaded)

    if not zipfile.is_zipfile(downloaded_path):
        raise RuntimeError(
            "The Drive file is not a valid ZIP archive. Zip the dataset folder and share the ZIP file link."
        )

    if not any(DRIVE_DOWNLOAD_DIR.iterdir()):
        print(f"[INFO] Extracting Google Drive ZIP to: {DRIVE_DOWNLOAD_DIR}")

        with zipfile.ZipFile(downloaded_path, "r") as zip_ref:
            zip_ref.extractall(DRIVE_DOWNLOAD_DIR)


def download_small_drive_folder(gdown, url):
    if not ALLOW_SMALL_DRIVE_FOLDER_DOWNLOAD:
        raise RuntimeError(
            "The provided Google Drive link is a folder. gdown cannot download large Drive folders "
            "with thousands of files. Zip the Drive folder first, share the ZIP file, then paste its "
            "link into DRIVE_ZIP_URL. Keep DRIVE_FOLDER_URL only as a reference."
        )

    DRIVE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    try:
        print(f"[INFO] Downloading small Google Drive folder to: {DRIVE_DOWNLOAD_DIR}")
        gdown.download_folder(
            url=url,
            output=str(DRIVE_DOWNLOAD_DIR),
            quiet=False
        )
    except Exception as error:
        message = str(error)
        class_name = error.__class__.__name__

        if "FolderContentsMaximumLimitError" in class_name or "more than 50 files" in message:
            raise RuntimeError(
                "Google Drive folder has too many files for gdown.download_folder. "
                "Zip the folder and put the ZIP file link in DRIVE_ZIP_URL."
            ) from error

        raise


def download_drive_dataset():
    if DRIVE_DATASET_ROOT is not None:
        drive_root = Path(DRIVE_DATASET_ROOT)
        detected_root = find_imagefolder_inside(drive_root, min_classes=1)

        if detected_root is not None:
            return detected_root

        raise FileNotFoundError(f"DRIVE_DATASET_ROOT is not an ImageFolder dataset: {drive_root}")

    if not DRIVE_ZIP_URL and not DRIVE_FOLDER_URL:
        return None

    gdown = import_gdown()

    if DRIVE_ZIP_URL:
        download_drive_zip(gdown, DRIVE_ZIP_URL)
    elif DRIVE_FOLDER_URL:
        download_small_drive_folder(gdown, DRIVE_FOLDER_URL)

    drive_root = find_imagefolder_inside(DRIVE_DOWNLOAD_DIR, min_classes=1)

    if drive_root is None:
        raise FileNotFoundError(
            "Downloaded Drive dataset was not found in ImageFolder format. "
            "Expected: root/class_name/image.jpg"
        )

    return drive_root


def safe_reset_directory(path):
    path = Path(path)
    working_root = WORKING_DIR.resolve()

    try:
        path.resolve().relative_to(working_root)
        is_safe = True
    except ValueError:
        is_safe = False

    if path.exists():
        if not is_safe:
            raise ValueError(f"Refusing to clear a directory outside WORKING_DIR: {path}")

        shutil.rmtree(path)

    path.mkdir(parents=True, exist_ok=True)


def link_or_copy_file(source_path, destination_path):
    source_path = Path(source_path)
    destination_path = Path(destination_path)

    if destination_path.exists():
        return

    if LINK_DATASETS_INSTEAD_OF_COPY:
        try:
            os.symlink(source_path, destination_path)
            return
        except OSError:
            pass

    shutil.copy2(source_path, destination_path)


def selected_classes_set():
    if SELECTED_CLASSES is None:
        return None

    return {str(class_name) for class_name in SELECTED_CLASSES}


def direct_image_files(folder):
    folder = Path(folder)

    for file_path in folder.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
            yield file_path


def iter_image_files(folder):
    folder = Path(folder)

    for file_path in folder.rglob("*"):
        if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
            yield file_path


def class_folder_has_images(folder):
    try:
        return next(direct_image_files(folder), None) is not None
    except PermissionError:
        return False


def dataset_build_config():
    return {
        "version": DATASET_BUILD_VERSION,
        "max_images_per_class": MAX_IMAGES_PER_CLASS,
        "selected_classes": None if SELECTED_CLASSES is None else sorted(selected_classes_set())
    }


def dataset_build_config_path(path):
    return Path(path) / "_build_config.json"


def combined_dataset_metadata_valid(path):
    metadata_path = dataset_build_config_path(path)

    if not metadata_path.exists():
        print("[INFO] Existing combined dataset has no build metadata; rebuilding.")
        return False

    try:
        with open(metadata_path, "r") as f:
            existing_config = json.load(f)
    except (json.JSONDecodeError, OSError):
        print("[INFO] Existing combined dataset metadata is invalid; rebuilding.")
        return False

    expected_config = dataset_build_config()

    if existing_config != expected_config:
        print("[INFO] Existing combined dataset config changed; rebuilding.")
        print(f"[INFO] Existing config: {existing_config}")
        print(f"[INFO] Expected config: {expected_config}")
        return False

    return True


def combined_dataset_within_image_limit(path):
    if not combined_dataset_metadata_valid(path):
        return False

    if MAX_IMAGES_PER_CLASS is None:
        return True

    if not is_imagefolder_root(path, min_classes=1):
        return False

    for class_dir in Path(path).iterdir():
        if not class_dir.is_dir():
            continue

        count = 0

        for _ in iter_image_files(class_dir):
            count += 1

            if count > MAX_IMAGES_PER_CLASS:
                print(
                    "[INFO] Existing combined dataset exceeds "
                    f"MAX_IMAGES_PER_CLASS={MAX_IMAGES_PER_CLASS}; rebuilding."
                )
                return False

    return True


def list_class_dirs(source_roots):
    class_dirs_by_name = {}
    allowed_classes = selected_classes_set()

    for source_name, source_root in source_roots:
        source_root = Path(source_root)

        for class_dir in sorted(source_root.iterdir()):
            if not class_dir.is_dir():
                continue

            class_name = class_dir.name

            if allowed_classes is not None and class_name not in allowed_classes:
                continue

            if not class_folder_has_images(class_dir):
                continue

            class_dirs_by_name.setdefault(class_name, []).append((source_name, class_dir))

    if allowed_classes is not None:
        missing_classes = sorted(allowed_classes - set(class_dirs_by_name.keys()))

        if missing_classes:
            print(f"[WARN] Selected classes not found: {missing_classes}")

    return class_dirs_by_name


def reservoir_sample_class(class_name, source_class_dirs):
    max_images = MAX_IMAGES_PER_CLASS
    stable_hash = hashlib.md5(f"{SEED}_{class_name}".encode("utf-8")).hexdigest()
    class_seed = int(stable_hash[:8], 16)
    rng = np.random.default_rng(class_seed)
    selected_samples = []
    total_seen = 0

    for source_name, class_dir in source_class_dirs:
        for image_path in iter_image_files(class_dir):
            total_seen += 1
            sample = (source_name, total_seen, image_path)

            if max_images is None or len(selected_samples) < max_images:
                selected_samples.append(sample)
            else:
                replace_idx = rng.integers(0, total_seen)

                if replace_idx < max_images:
                    selected_samples[replace_idx] = sample

    selected_samples.sort(key=lambda item: str(item[2]))

    print(
        f"[INFO] Class '{class_name}': "
        f"using {len(selected_samples)}/{total_seen} images"
    )

    return selected_samples


def add_selected_samples_to_combined(class_name, selected_samples, combined_root):
    destination_dir = Path(combined_root) / class_name
    destination_dir.mkdir(parents=True, exist_ok=True)

    for sample_idx, (source_name, seen_idx, image_path) in enumerate(selected_samples):
        suffix = image_path.suffix.lower() or ".jpg"
        destination_name = f"{source_name}_{sample_idx:08d}_{seen_idx:08d}{suffix}"
        destination_path = destination_dir / destination_name

        link_or_copy_file(image_path, destination_path)


def build_combined_dataset(source_roots):
    if (
        COMBINED_DATASET_ROOT.exists()
        and not REBUILD_COMBINED_DATASET
        and combined_dataset_within_image_limit(COMBINED_DATASET_ROOT)
    ):
        combined_dataset = ImageFolder(root=COMBINED_DATASET_ROOT)
        print(f"[INFO] Reusing combined dataset: {COMBINED_DATASET_ROOT}")
        print(f"[INFO] Combined classes: {len(combined_dataset.classes)}")
        print(f"[INFO] Combined images: {len(combined_dataset.samples)}")
        return COMBINED_DATASET_ROOT

    safe_reset_directory(COMBINED_DATASET_ROOT)

    class_dirs_by_name = list_class_dirs(source_roots)

    for class_name in sorted(class_dirs_by_name.keys()):
        selected_samples = reservoir_sample_class(
            class_name=class_name,
            source_class_dirs=class_dirs_by_name[class_name]
        )
        add_selected_samples_to_combined(
            class_name=class_name,
            selected_samples=selected_samples,
            combined_root=COMBINED_DATASET_ROOT
        )

    with open(dataset_build_config_path(COMBINED_DATASET_ROOT), "w") as f:
        json.dump(dataset_build_config(), f, indent=4)

    combined_dataset = ImageFolder(root=COMBINED_DATASET_ROOT)

    print(f"[INFO] Combined dataset root: {COMBINED_DATASET_ROOT}")
    print(f"[INFO] Combined classes: {len(combined_dataset.classes)}")
    print(f"[INFO] Combined images: {len(combined_dataset.samples)}")

    return COMBINED_DATASET_ROOT

def prepare_dataset_root():
    if DATASET_ROOT is not None:
        dataset_root = Path(DATASET_ROOT)
        detected_root = find_imagefolder_inside(dataset_root, min_classes=1)

        if detected_root is None:
            raise FileNotFoundError(f"DATASET_ROOT is not an ImageFolder dataset: {dataset_root}")

        return detected_root

    source_roots = []

    prepare_local_kaggle_cache()

    kaggle_roots = find_kaggle_dataset_roots()

    if kaggle_roots:
        for dataset_idx, kaggle_root in enumerate(kaggle_roots, start=1):
            print(f"[INFO] Kaggle dataset root {dataset_idx}: {kaggle_root}")
            source_roots.append((f"kaggle_{dataset_idx}", kaggle_root))
    else:
        print("[WARN] No Kaggle ImageFolder dataset detected.")

    drive_root = download_drive_dataset()

    if drive_root is not None:
        print(f"[INFO] Google Drive dataset root: {drive_root}")
        source_roots.append(("drive", drive_root))
    else:
        print("[INFO] No Google Drive dataset URL configured.")

    if not source_roots:
        raise FileNotFoundError(
            "No dataset found. Add a Kaggle dataset with Add Input, set DRIVE_ZIP_URL, "
            "or set DATASET_ROOT manually."
        )

    if len(source_roots) == 1 and not COMBINE_DATASETS and MAX_IMAGES_PER_CLASS is None:
        return source_roots[0][1]

    return build_combined_dataset(source_roots)

In [ ]:
# =========================================================
# MODEL HELPERS
# =========================================================


def create_effnetb2_model(num_classes, seed=42, use_pretrained=True):
    torch.manual_seed(seed)

    fallback_transforms = torchvision.transforms.Compose([
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    if use_pretrained:
        try:
            weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
            transforms = weights.transforms()
            model = torchvision.models.efficientnet_b2(weights=weights)
            print("[INFO] Loaded ImageNet pretrained EfficientNet_B2 weights.")
        except Exception as error:
            print(f"[WARN] Could not load pretrained weights: {error}")
            print("[WARN] Training EfficientNet_B2 from scratch. Enable Kaggle internet for better results.")
            transforms = fallback_transforms
            model = torchvision.models.efficientnet_b2(weights=None)
    else:
        transforms = fallback_transforms
        model = torchvision.models.efficientnet_b2(weights=None)

    for param in model.parameters():
        param.requires_grad = False

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1408, out_features=num_classes)
    )

    return model, transforms


def strip_module_prefix(state_dict):
    return {
        key.replace("module.", "", 1): value
        for key, value in state_dict.items()
    }


def load_model_if_available(model, model_path):
    if not model_path.exists():
        print("[INFO] No saved model found. Training from scratch.")
        return model

    print(f"[INFO] Loading existing model: {model_path}")
    checkpoint = torch.load(model_path, map_location="cpu")

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]

    checkpoint = strip_module_prefix(checkpoint)
    model.load_state_dict(checkpoint)

    return model


def get_base_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def save_training_plot(values, labels, title, ylabel, output_path):
    plt.figure(figsize=(12, 5))

    for value, label in zip(values, labels):
        plt.plot(epochs, value, label=label)

    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"[INFO] Saved plot: {output_path}")

In [ ]:
# =========================================================
# PREPARE DATASET, MODEL, AND DATALOADERS
# =========================================================

set_seeds(SEED)

food_root = prepare_dataset_root()
print(f"[INFO] Final dataset root: {food_root}")

dataset_without_transform = ImageFolder(root=food_root)
class_names = dataset_without_transform.classes

print(f"[INFO] Classes: {len(class_names)}")
print(f"[INFO] Total images: {len(dataset_without_transform)}")

base_model, effnetb2_transforms = create_effnetb2_model(
    num_classes=len(class_names),
    seed=SEED,
    use_pretrained=True
)

base_model = load_model_if_available(base_model, MODEL_PATH)

for param in base_model.features[-FINE_TUNE_LAST_BLOCKS:].parameters():
    param.requires_grad = True

print(f"[INFO] Fine-tuning last {FINE_TUNE_LAST_BLOCKS} EfficientNet feature blocks.")

if RUN_MODEL_SUMMARY and summary is not None:
    try:
        print("\n[INFO] Model Summary:")
        summary(
            base_model,
            input_size=(1, 3, 224, 224),
            device="cpu"
        )
    except Exception as error:
        print(f"[WARN] Model summary skipped: {error}")

dataset = ImageFolder(
    root=food_root,
    transform=effnetb2_transforms
)

train_size = int(TRAIN_RATIO * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS
)

print(f"[INFO] Train images: {len(train_dataset)}")
print(f"[INFO] Test images: {len(test_dataset)}")
print(f"[INFO] Train batches: {len(train_loader)}")
print(f"[INFO] Test batches: {len(test_loader)}")

effnetb2 = base_model.to(device)

if NUM_GPUS > 1:
    print(f"[INFO] Enabling DataParallel on {NUM_GPUS} GPUs.")
    effnetb2 = nn.DataParallel(effnetb2)

optimizer = torch.optim.Adam(
    effnetb2.parameters(),
    lr=LEARNING_RATE
)

loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler(AMP_DEVICE_TYPE, enabled=USE_AMP)

if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r") as f:
        history = json.load(f)

    print("[INFO] Loaded previous history")
else:
    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
        "f1_score": [],
        "precision": [],
        "recall": []
    }

    print("[INFO] Starting new history")

results = {
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": [],
    "f1_score": [],
    "precision": [],
    "recall": []
}

In [ ]:
# =========================================================
# TRAINING
# =========================================================

print("[INFO] Training model...")

best_test_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    print(f"\n========== Epoch {epoch + 1}/{EPOCHS} ==========")

    effnetb2.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for X, y in train_loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=AMP_DEVICE_TYPE,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            y_pred = effnetb2(X)
            loss = loss_fn(y_pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        preds = torch.argmax(y_pred, dim=1)
        train_correct += (preds == y).sum().item()
        train_total += len(y)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total

    effnetb2.eval()

    test_loss = 0
    test_correct = 0
    test_total = 0

    all_preds = []
    all_labels = []

    with torch.inference_mode():
        for X, y in test_loader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.autocast(
                device_type=AMP_DEVICE_TYPE,
                dtype=torch.float16,
                enabled=USE_AMP
            ):
                test_logits = effnetb2(X)
                loss = loss_fn(test_logits, y)

            test_loss += loss.item()

            preds = torch.argmax(test_logits, dim=1)
            test_correct += (preds == y).sum().item()
            test_total += len(y)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    test_loss /= len(test_loader)
    test_acc = test_correct / test_total

    f1 = f1_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    precision = precision_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)
    results["f1_score"].append(f1)
    results["precision"].append(precision)
    results["recall"].append(recall)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Test Loss  : {test_loss:.4f}")
    print(f"Test Acc   : {test_acc:.4f}")
    print(f"F1 Score   : {f1:.4f}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")

    if test_loss < best_test_loss - EARLY_STOPPING_MIN_DELTA:
        best_test_loss = test_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(
            get_base_model(effnetb2).state_dict(),
            MODEL_PATH
        )

        print(f"[INFO] Best model updated at epoch {best_epoch}: test_loss={best_test_loss:.4f}")
    else:
        epochs_without_improvement += 1
        print(
            f"[INFO] No improvement for {epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE} epochs."
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"[INFO] Early stopping at epoch {epoch + 1}.")
            break

for key in history.keys():
    history[key].extend(results[key])

with open(HISTORY_PATH, "w") as f:
    json.dump(history, f, indent=4)

print(f"\n[INFO] History saved to {HISTORY_PATH}")

if MODEL_PATH.exists():
    print(f"[INFO] Best model saved to {MODEL_PATH} from epoch {best_epoch}")
else:
    torch.save(
        get_base_model(effnetb2).state_dict(),
        MODEL_PATH
    )
    print(f"[INFO] Model saved to {MODEL_PATH}")

epochs = range(1, len(history["train_loss"]) + 1)

save_training_plot(
    values=[history["train_loss"], history["test_loss"]],
    labels=["Train Loss", "Test Loss"],
    title="Loss Curve",
    ylabel="Loss",
    output_path=PLOT_DIR / "loss_curve.png"
)

save_training_plot(
    values=[history["train_acc"], history["test_acc"]],
    labels=["Train Accuracy", "Test Accuracy"],
    title="Accuracy Curve",
    ylabel="Accuracy",
    output_path=PLOT_DIR / "accuracy_curve.png"
)

save_training_plot(
    values=[history["f1_score"]],
    labels=["F1 Score"],
    title="F1 Score Curve",
    ylabel="F1 Score",
    output_path=PLOT_DIR / "f1_score_curve.png"
)

print("\n[INFO] Training complete!")

In [ ]:
def make_confusion_matrix(
    y_true,
    y_pred,
    classes=None,
    figsize=(20, 20),
    text_size=8,
    norm=False,
    savefig=False
):

    if classes:
        labels = classes
        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=np.arange(len(classes))
        )
    else:
        labels = np.arange(len(np.unique(y_true)))
        cm = confusion_matrix(y_true, y_pred)

    with np.errstate(divide="ignore", invalid="ignore"):
        cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

    n_classes = cm.shape[0]

    fig, ax = plt.subplots(figsize=figsize)
    cax = ax.matshow(cm, cmap=plt.cm.Blues)
    fig.colorbar(cax)

    ax.set(
        title="Confusion Matrix",
        xlabel="Predicted Label",
        ylabel="True Label",
        xticks=np.arange(n_classes),
        yticks=np.arange(n_classes),
        xticklabels=labels,
        yticklabels=labels
    )

    ax.xaxis.set_label_position("bottom")
    ax.xaxis.tick_bottom()
    plt.xticks(rotation=90)

    threshold = (cm.max() + cm.min()) / 2.
    annotate_cells = n_classes <= MAX_CONFUSION_ANNOTATION_CLASSES

    if not annotate_cells:
        print(
            f"[INFO] Skipping per-cell annotations for {n_classes} classes "
            f"to save memory/time."
        )

    if annotate_cells:
        for i, j in itertools.product(
            range(cm.shape[0]),
            range(cm.shape[1])
        ):
            if norm:
                text = f"{cm[i, j]}\n({cm_norm[i, j] * 100:.1f}%)"
            else:
                text = f"{cm[i, j]}"

            plt.text(
                j,
                i,
                text,
                horizontalalignment="center",
                color="white" if cm[i, j] > threshold else "black",
                size=text_size
            )

    plt.tight_layout()

    if savefig:
        output_dir = globals().get("PLOT_DIR", Path("."))
        output_path = output_dir / "confusion_matrix.png"

        plt.savefig(
            output_path,
            dpi=300,
            bbox_inches="tight"
        )

        print(f"[INFO] Confusion matrix saved to {output_path}")

    plt.show()

In [ ]:
# =========================================================
# CONFUSION MATRIX + ROC/AUC
# =========================================================

PLOT_DIR = globals().get("PLOT_DIR", Path("."))
MODEL_DIR = globals().get("MODEL_DIR", Path("models"))
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def sample_roc_inputs(y_true, y_score, max_samples, seed=42):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    if max_samples is None or len(y_true) <= max_samples:
        return y_true, y_score

    rng = np.random.default_rng(seed)
    selected_indices = rng.choice(len(y_true), size=max_samples, replace=False)
    selected_indices.sort()

    print(
        f"[INFO] Sampling {max_samples}/{len(y_true)} test predictions "
        "for ROC/AUC to save memory."
    )

    return y_true[selected_indices], y_score[selected_indices]


def plot_multiclass_roc_auc(
    y_true,
    y_score,
    classes,
    savefig=True
):

    n_classes = len(classes)
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    if len(np.unique(y_true)) < 2:
        print("[WARN] ROC/AUC needs at least two classes in y_true.")
        return {}

    plt.figure(figsize=(12, 10))

    if n_classes == 2:
        positive_class = 1
        y_binary = (y_true == positive_class).astype(int)

        fpr, tpr, _ = roc_curve(
            y_binary,
            y_score[:, positive_class]
        )

        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            color="darkorange",
            lw=2,
            label=f"ROC curve (AUC = {roc_auc:.3f})"
        )

        auc_scores = {
            classes[positive_class]: float(roc_auc)
        }

        print("\n[INFO] ROC/AUC results:")
        print(f"AUC ({classes[positive_class]}): {roc_auc:.4f}")

    else:
        y_true_bin = label_binarize(
            y_true,
            classes=np.arange(n_classes)
        )

        fpr = {}
        tpr = {}
        roc_auc = {}
        valid_class_indices = []

        for class_idx in range(n_classes):
            if len(np.unique(y_true_bin[:, class_idx])) < 2:
                continue

            fpr[class_idx], tpr[class_idx], _ = roc_curve(
                y_true_bin[:, class_idx],
                y_score[:, class_idx]
            )

            roc_auc[class_idx] = auc(
                fpr[class_idx],
                tpr[class_idx]
            )

            valid_class_indices.append(class_idx)

        if not valid_class_indices:
            print("[WARN] ROC/AUC could not be computed.")
            plt.close()
            return {}

        fpr["micro"], tpr["micro"], _ = roc_curve(
            y_true_bin[:, valid_class_indices].ravel(),
            y_score[:, valid_class_indices].ravel()
        )

        roc_auc["micro"] = auc(
            fpr["micro"],
            tpr["micro"]
        )

        all_fpr = np.unique(
            np.concatenate(
                [
                    fpr[class_idx]
                    for class_idx in valid_class_indices
                ]
            )
        )

        mean_tpr = np.zeros_like(all_fpr)

        for class_idx in valid_class_indices:
            mean_tpr += np.interp(
                all_fpr,
                fpr[class_idx],
                tpr[class_idx]
            )

        mean_tpr /= len(valid_class_indices)

        fpr["macro"] = all_fpr
        tpr["macro"] = mean_tpr

        roc_auc["macro"] = auc(
            fpr["macro"],
            tpr["macro"]
        )

        show_class_legend = len(valid_class_indices) <= 15

        colors = plt.cm.tab20(
            np.linspace(
                0,
                1,
                max(len(valid_class_indices), 1)
            )
        )

        for class_idx, color in zip(valid_class_indices, colors):
            label = (
                f"{classes[class_idx]} (AUC = {roc_auc[class_idx]:.2f})"
                if show_class_legend
                else None
            )

            plt.plot(
                fpr[class_idx],
                tpr[class_idx],
                color=color,
                lw=1,
                alpha=0.35,
                label=label
            )

        if not show_class_legend:
            plt.plot(
                [],
                [],
                color="gray",
                lw=1,
                alpha=0.35,
                label="One-vs-rest class ROC curves"
            )

        plt.plot(
            fpr["micro"],
            tpr["micro"],
            color="deeppink",
            linestyle=":",
            linewidth=4,
            label=f"Micro-average ROC (AUC = {roc_auc['micro']:.3f})"
        )

        plt.plot(
            fpr["macro"],
            tpr["macro"],
            color="navy",
            linestyle=":",
            linewidth=4,
            label=f"Macro-average ROC (AUC = {roc_auc['macro']:.3f})"
        )

        auc_scores = {
            "micro_average": float(roc_auc["micro"]),
            "macro_average": float(roc_auc["macro"])
        }

        auc_scores.update(
            {
                classes[class_idx]: float(roc_auc[class_idx])
                for class_idx in valid_class_indices
            }
        )

        print("\n[INFO] ROC/AUC results:")
        print(f"Micro-average AUC: {roc_auc['micro']:.4f}")
        print(f"Macro-average AUC: {roc_auc['macro']:.4f}")

        for class_idx in valid_class_indices:
            print(f"{classes[class_idx]} AUC: {roc_auc[class_idx]:.4f}")

    plt.plot(
        [0, 1],
        [0, 1],
        color="black",
        linestyle="--",
        lw=1
    )

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves and AUC")
    plt.legend(loc="lower right", fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()

    if savefig:
        roc_curve_path = PLOT_DIR / "roc_auc_curve.png"

        plt.savefig(
            roc_curve_path,
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()

    roc_scores_path = MODEL_DIR / "roc_auc_scores.json"

    with open(roc_scores_path, "w") as f:
        json.dump(auc_scores, f, indent=4)

    if savefig:
        print(f"[INFO] ROC curve saved to {roc_curve_path}")

    print(f"[INFO] AUC scores saved to {roc_scores_path}")

    return auc_scores


print("[INFO] Creating confusion matrix and evaluation curves...")

effnetb2.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.inference_mode():
    for X, y in test_loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.autocast(
            device_type=AMP_DEVICE_TYPE,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            logits = effnetb2(X)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

make_confusion_matrix(
    y_true=all_labels,
    y_pred=all_preds,
    classes=class_names,
    figsize=(30, 30),
    text_size=6,
    norm=True,
    savefig=True
)

if COMPUTE_ROC_AUC:
    roc_labels, roc_probs = sample_roc_inputs(
        y_true=all_labels,
        y_score=all_probs,
        max_samples=ROC_MAX_SAMPLES,
        seed=SEED
    )

    roc_auc_scores = plot_multiclass_roc_auc(
        y_true=roc_labels,
        y_score=roc_probs,
        classes=class_names,
        savefig=True
    )
else:
    roc_auc_scores = {}
    print("[INFO] ROC/AUC skipped because COMPUTE_ROC_AUC=False.")